# Phân tích các mô hình đã huấn luyện

Notebook này chỉ đọc checkpoint đã có để chạy thêm phần đánh giá và giải thích, không huấn luyện lại mô hình

Các phần chính gồm:

- Kiểm tra trên NF-ToN-IoT-v3, một bộ dữ liệu không dùng khi huấn luyện
- Xem đặc trưng nào ảnh hưởng nhiều đến dự đoán
- Tìm vùng mạng và các cạnh liên quan đến cảnh báo
- Gom kết quả cùng phân phối và ngoài phân phối vào một bảng

Các phép đánh giá cần đầu ra phân loại nên chỉ dùng checkpoint stage 2; stage 1 là bộ mã hoá tiền huấn luyện nên không có ngưỡng dự đoán nhị phân để đối sánh

Ở phép kiểm ngoài phân phối, thứ tự dòng đang được dùng làm biến trình tự nên kết quả không đại diện cho một lát cắt theo thời gian tuyệt đối


## 1. Chuẩn bị môi trường

Notebook chạy tại máy nên đường dẫn dựng từ vị trí của chính tệp notebook, không
dùng đường dẫn tuyệt đối của một hệ điều hành cụ thể. Dự án được mở luân phiên
trên nhiều hệ điều hành và cách dựng đường dẫn này giữ cho notebook chạy được
trên mọi hệ mà không phải sửa tay.


In [1]:
import os
import sys
import json
import time
import glob
import zlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Dựng đường dẫn gốc từ thư mục làm việc hiện tại. Notebook nằm ở
# notebooks/<bản thực nghiệm>/ nên thư mục gốc dự án cách hai cấp; kiểm sự tồn
# tại của thư mục src để xác nhận đã tìm đúng chứ không đếm số cấp một cách máy
# móc, vì Jupyter có thể được khởi động từ nhiều thư mục khác nhau.
PROJECT_ROOT = Path.cwd()
for _ in range(3):
    if (PROJECT_ROOT / "src" / "tgn_nids").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc dự án. Khởi động Jupyter từ trong thư mục "
        "dự án hoặc từ một thư mục con của nó.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import tgn_nids
from tgn_nids.experiments.ood_eval import (evaluate_ood_binary,
                                           reservoir_sample_parquet)
from tgn_nids.data.preprocess import NetFlowPreprocessor
from tgn_nids.utils import figures as fig_lib

# Đặt kiểu vẽ chung: phông chữ Times New Roman, đơn sắc xám cộng một màu nhấn.
fig_lib.apply_style()

# In tên thư mục thôi, không in đường dẫn tuyệt đối: notebook này là tệp nộp,
# và đường dẫn tuyệt đối để lộ tên tài khoản cùng bố cục ổ đĩa của máy đã chạy.
print(f"Thư mục gốc dự án : {PROJECT_ROOT.name}")
print(f"Phiên bản mã nguồn: {tgn_nids.SOURCE_VERSION}")
print(f"Python            : {sys.version.split()[0]}")
print(f"PyTorch           : {torch.__version__}")

Thư mục gốc dự án : TGN-NIDS
Phiên bản mã nguồn: 1.1.0
Python            : 3.13.5
PyTorch           : 2.8.0


## 2. Bố cục thư mục kết quả

Kết quả của notebook này nằm trong một nhóm riêng, tách khỏi kết quả huấn luyện.
Lý do là quy trình thu gom kết quả từ nền tảng từ xa xoá sạch thư mục nhóm trước
khi chép nội dung mới; sản phẩm sinh tại máy nếu để chung sẽ mất trong lần thu
gom kế tiếp.


In [2]:
# Bản thực nghiệm đang phân tích. Đổi hằng số này để chuyển sang bản còn lại;
# mọi đường dẫn phía dưới bám theo nó.
VARIANT = "twoDTS"

# Hai thư mục kết quả của hai notebook huấn luyện, đọc thẳng theo đúng bố cục
# <NOTEBOOK_ID>/<ô mã>/models mà chúng sinh ra.
#
# Đọc thẳng từ nơi notebook huấn luyện ghi ra, không qua một thư mục trung gian
# nào. Bất kỳ bước gom tay nào ở giữa cũng là một bước có thể quên, và quên thì
# ô nạp mô hình trả về danh sách rỗng chứ mà không phát cảnh báo.
TRAINED_ROOTS = [
    PROJECT_ROOT / "models" / "saved" / f"{VARIANT}_train_v2",
    PROJECT_ROOT / "models" / "saved" / f"{VARIANT}_train_v3",
]
# Nhóm chứa sản phẩm của chính notebook này.
OUTPUT_ROOT = PROJECT_ROOT / "models" / "saved" / f"{VARIANT}_phan_tich"
SUBFOLDERS = ("metrics", "tables", "figures")


def cell_dir(cell_name, subfolder):
    """Trả về thư mục riêng của một ô mã, tạo trước nếu chưa tồn tại."""
    # Chặn tên thư mục con viết sai ngay tại đây; nếu để lọt, tệp sẽ nằm ở một
    # nhánh không nằm trong phạm vi thu gom kết quả.
    if subfolder not in SUBFOLDERS:
        raise ValueError(f"Thư mục con không hợp lệ: {subfolder}")
    path = OUTPUT_ROOT / subfolder / cell_name
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
def relative_to_project(path):
    """Rút gọn đường dẫn về dạng tương đối so với gốc dự án, để in ra ngoài.

    Notebook này là tệp nộp, nên đầu ra không được chứa đường dẫn tuyệt đối:
    đường dẫn tuyệt đối lộ tên tài khoản và bố cục ổ đĩa của máy đã chạy, đồng
    thời làm đầu ra khác nhau giữa hai máy dù kết quả hoàn toàn giống nhau.
    """
    return path.relative_to(PROJECT_ROOT)


for root in TRAINED_ROOTS:
    print(f"Mô hình đã huấn luyện: {relative_to_project(root)}  "
          f"({'có' if root.is_dir() else 'chưa có'})")
print(f"Kết quả phân tích    : {relative_to_project(OUTPUT_ROOT)}")

Mô hình đã huấn luyện: models/saved/twoDTS_train_v2  (có)
Mô hình đã huấn luyện: models/saved/twoDTS_train_v3  (có)
Kết quả phân tích    : models/saved/twoDTS_phan_tich


## 3. Nạp mô hình đã huấn luyện

Phần mã kế tiếp dò toàn bộ tệp trọng số trong nhóm kết quả huấn luyện và đọc bản
kê khai chỉ số đi kèm mỗi tệp. Mỗi mô hình được kiểm ba điều kiện trước khi đưa
vào phân tích: tệp trọng số tồn tại, tệp chỉ số tồn tại, và phiên bản mã nguồn
ghi trong kết quả trùng với phiên bản đang chạy.

Điều kiện thứ ba đề phòng trường hợp mã nguồn đã đổi sau khi mô hình được huấn
luyện. Khi đó cấu trúc mô hình có thể không còn khớp với trọng số đã lưu, và
việc nạp sẽ thành công một phần rồi cho ra kết quả sai mà không báo lỗi.


In [3]:
CELL = "01_nap_mo_hinh"

available_roots = [root for root in TRAINED_ROOTS if root.is_dir()]
if not available_roots:
    raise FileNotFoundError(
        f"Không tìm thấy thư mục kết quả nào trong {[str(r) for r in TRAINED_ROOTS]}. "
        f"Chạy quy trình thu gom kết quả từ nền tảng tính toán từ xa trước khi "
        f"chạy notebook này.")

# Quét cả hai thư mục, mỗi thư mục theo bố cục <ô mã>/models. Sắp theo tên lần
# chạy để thứ tự không phụ thuộc thứ tự duyệt thư mục của hệ thống tệp.
result_paths = sorted(
    (path for root in available_roots for path in root.glob("*/models/result_*.json")),
    key=lambda path: path.name)

models = {}
load_rows = []
for result_path in result_paths:
    run_name = result_path.name[len("result_"):-len(".json")]
    model_path = result_path.parent / f"model_{run_name}.pt"

    # Bỏ qua lần chạy thiếu tệp trọng số thay vì dừng cả notebook: một lần chạy
    # hỏng không nên chặn phần phân tích của các lần chạy còn lại.
    if not model_path.exists():
        print(f"Bỏ qua {run_name}: thiếu tệp trọng số")
        continue

    with open(result_path, encoding="utf-8") as handle:
        result = json.load(handle)

    # Đối chiếu phiên bản mã nguồn. Lệch phiên bản nghĩa là mô hình được huấn
    # luyện bằng một bản mã khác bản đang chạy.
    saved_version = result.get("source_version")
    if saved_version != tgn_nids.SOURCE_VERSION:
        raise RuntimeError(
            f"Lần chạy {run_name} sinh ra bởi mã nguồn phiên bản {saved_version}, "
            f"còn phiên bản đang chạy là {tgn_nids.SOURCE_VERSION}. Cấu trúc mô "
            f"hình có thể đã đổi, nên việc nạp trọng số không an toàn.")

    models[run_name] = {"model_path": str(model_path), "result": result}
    metrics = result["metrics"]
    load_rows.append({
        "run_name": run_name,
        "task": result["config"]["task"],
        "mode": result["config"]["mode"],
        "f1_macro": metrics.get("f1_macro"),
        "accuracy": metrics.get("accuracy"),
        "best_epoch": result.get("best_epoch"),
        "n_parameters": result.get("n_parameters"),
        "source_version": saved_version,
    })

if not models:
    raise RuntimeError(
        f"Không nạp được mô hình nào từ {[str(r) for r in available_roots]}")

# Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
# giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
# vào tệp nộp.
_written_path = fig_lib.export_table(load_rows, cell_dir(CELL, "tables"), "mo_hinh_da_nap")

print(f"{'Lần chạy':<38}{'Bài toán':<12}{'Chế độ':<18}{'Macro F1':>10}")
for row in load_rows:
    print(f"{row['run_name']:<38}{row['task']:<12}{row['mode']:<18}"
          f"{row['f1_macro'] * 100:>9.3f}%")

Lần chạy                              Bài toán    Chế độ              Macro F1
stage1_nf_cse_cic_ids2018_v3          binary      self_supervised      55.556%
stage1_nf_unsw_nb15_v2                binary      self_supervised      82.906%
stage1_nf_unsw_nb15_v3                binary      self_supervised      98.695%
stage2_nf_cse_cic_ids2018_v3          multiclass  supervised           56.372%
stage2_nf_unsw_nb15_v2                multiclass  supervised           53.340%
stage2_nf_unsw_nb15_v3                multiclass  supervised           55.961%


## 4. Đánh giá khả năng khái quát out-of-distribution

Bộ NF-ToN-IoT-v3 được đưa qua mô hình đã huấn luyện. Đây là lưu lượng của môi
trường Internet vạn vật, khác hẳn môi trường đã sinh ra dữ liệu huấn luyện, và
không tham gia vào bất kỳ bước huấn luyện nào.

**Vì sao chỉ một bộ.** Phương pháp dựng đồ thị từ địa chỉ máy, nên bộ dữ liệu
ngoài bắt buộc phải có cột địa chỉ nguồn và đích. Bản NF-ToN-IoT phiên bản đặc
trưng thứ hai có 43 cột và không kèm cột địa chỉ nào, nên `evaluate_ood_binary`
dừng với lỗi thay vì sinh địa chỉ thay thế: định danh đỉnh quyết định toàn bộ
cấu trúc đồ thị và sẽ tham gia vào mọi chỉ số được báo cáo, nên một đồ thị dựng
trên định danh bịa vẫn chạy trơn tru và cho ra bảng chỉ số không có nghĩa.

**Cách lấy mẫu.** Bộ dữ liệu được rút 250 nghìn luồng bằng phương pháp hồ chứa
giữ nguyên tỷ lệ nhãn của bộ gốc. Không lấy phần đầu tệp vì các bộ NetFlow được
sắp theo thời gian thu thập, nên phần đầu chỉ chứa vài kiểu tấn công đầu tiên
và không đại diện cho toàn bộ.

**Số đặc trưng thiếu** được ghi lại. Mô hình huấn luyện trên phiên
bản đặc trưng nào thì chỉ nhận đủ đầu vào từ bộ dữ liệu cùng phiên bản đó; cột
vắng mặt bị điền giá trị thay thế, và mọi chỉ số đo trong điều kiện ấy phải đọc
kèm số cột đã điền.

**Giao thức chọn ngưỡng.** Bộ dữ liệu được chia đôi giữ nguyên tỷ lệ nhãn. Ngưỡng
chọn trên nửa thứ nhất theo tiêu chí cực đại điểm F1, rồi áp lên nửa thứ hai.
Chỉ số báo cáo lấy từ nửa thứ hai.


In [4]:
CELL = "02_ngoai_phan_phoi"

# Bộ dữ liệu ngoài phân phối, dùng để đo mô hình trên một môi trường mạng khác
# hẳn môi trường nó được huấn luyện. Đường dẫn dựng từ thư mục gốc dự án nên
# chạy được trên mọi hệ điều hành.
#
# Chỉ dùng bản phiên bản đặc trưng thứ ba. Phép đo này dựng lại đồ thị máy chủ
# trên dữ liệu mới nên bộ dữ liệu bắt buộc phải có cột địa chỉ; bản NF-ToN-IoT
# phiên bản 2 không có cột nào như vậy. Hàm đánh giá ngoài phân phối dừng với
# lỗi khi thiếu địa chỉ, thay vì suy ra định danh đỉnh từ một cột khác: một
# topology suy ra như vậy không tồn tại trong dữ liệu, và mọi chỉ số đo trên nó
# đều nói về một mạng không có thật.
OOD_DATASETS = {
    "NF-ToN-IoT-v3": PROJECT_ROOT / "data/raw/nf-ton-iot-v3/NF-ToN-IoT-v3.parquet",
}

TARGET_ROWS = 250_000
SAMPLE_SEED = 42
SPLIT_SEED = 0


def f1_at_threshold(labels, scores, threshold):
    """Tính F1, precision và recall tại một ngưỡng cụ thể.

    Tự tính từ ma trận nhầm lẫn thay vì gọi thư viện vì hàm này được gọi vài
    nghìn lần trong mỗi lần quét ngưỡng.
    """
    predicted = scores >= threshold
    positive = labels == 1
    true_positive = int(np.sum(predicted & positive))
    false_positive = int(np.sum(predicted & ~positive))
    false_negative = int(np.sum(~predicted & positive))
    # Không có dự đoán dương nào thì precision không xác định; trả về 0 thay vì
    # để phép chia cho 0 sinh ra giá trị không phải số lan sang các bước sau.
    if true_positive == 0:
        return 0.0, 0.0, 0.0
    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    return 2 * precision * recall / (precision + recall), precision, recall


def best_threshold(labels, scores, n_points=2001):
    """Tìm ngưỡng cho F1 cao nhất, quét trên các phân vị của chính điểm số.

    Quét theo phân vị thay vì chia đều đoạn từ 0 tới 1: điểm số trên dữ liệu lạ
    thường dồn sát hai đầu, nên chia đều sẽ đặt gần hết số điểm quét vào vùng
    không có mẫu nào trong khi vùng đông mẫu chỉ được vài điểm.
    """
    candidates = np.unique(np.quantile(scores, np.linspace(0.0, 1.0, n_points)))
    best_value, best_score = float(candidates[0]), -1.0
    for threshold in candidates:
        score = f1_at_threshold(labels, scores, threshold)[0]
        if score > best_score:
            best_value, best_score = float(threshold), score
    return best_value


def split_in_half(labels, seed=SPLIT_SEED):
    """Chia đôi chỉ số mẫu, giữ nguyên tỷ lệ nhãn ở cả hai nửa.

    Xáo trộn riêng từng lớp rồi mới cắt đôi. Xáo trộn toàn bộ rồi cắt sẽ khiến
    tỷ lệ nhãn hai nửa lệch nhau ngẫu nhiên, mà mốc của bộ phân loại tầm thường
    lại phụ thuộc trực tiếp vào tỷ lệ ấy.
    """
    generator = np.random.default_rng(seed)
    positive = np.flatnonzero(labels == 1)
    negative = np.flatnonzero(labels == 0)
    generator.shuffle(positive)
    generator.shuffle(negative)
    half_a = np.concatenate([positive[:len(positive) // 2],
                             negative[:len(negative) // 2]])
    half_b = np.concatenate([positive[len(positive) // 2:],
                             negative[len(negative) // 2:]])
    return half_a, half_b


# Chỉ đánh giá ngoài phân phối cho mô hình đa lớp: đây là mô hình mà báo cáo
# trình bày, và điểm bất thường của nó định nghĩa được cho mọi luồng.
ANALYSIS_RUNS = [name for name in models if name.startswith("stage2_")]

ood_results = {}
ood_rows = []
for run_name in ANALYSIS_RUNS:
    model_path = models[run_name]["model_path"]
    for dataset_name, dataset_path in OOD_DATASETS.items():
        if not dataset_path.exists():
            print(f"Bỏ qua {dataset_name}: không tìm thấy {dataset_path.name}")
            continue

        started = time.time()
        external = reservoir_sample_parquet(str(dataset_path),
                                            target_rows=TARGET_ROWS,
                                            seed=SAMPLE_SEED)
        sample_seconds = time.time() - started

        started = time.time()
        # return_scores giữ lại điểm bất thường của từng luồng. Không có nó thì
        # chỉ nhận được chỉ số tổng hợp, và không tách được hai nguyên nhân của
        # một kết quả kém: ngưỡng đặt sai, hay mô hình không phân biệt được.
        #
        # Bắt ValueError để một bộ dữ liệu không dựng được đồ thị chỉ làm mất
        # hàng của chính nó, không làm gián đoạn các bộ còn lại. Bỏ qua rồi in lý do vẫn
        # tránh phải điền một hàng dựng trên đỉnh sinh ra.
        try:
            metrics = evaluate_ood_binary(model_path, external, device="cpu",
                                          return_scores=True)
        except ValueError as reason:
            print(f"Bỏ qua {run_name} trên {dataset_name}: {reason}")
            continue
        inference_seconds = time.time() - started

        labels = np.asarray(metrics["labels"]).astype(int)
        scores = np.asarray(metrics["scores"]).astype(float)
        positive_rate = float(labels.mean())
        trivial = fig_lib.trivial_f1(positive_rate)

        half_a, half_b = split_in_half(labels)
        tuned = best_threshold(labels[half_a], scores[half_a])
        f1_tuned, precision, recall = f1_at_threshold(labels[half_b],
                                                      scores[half_b], tuned)
        # Ngưỡng mặc định 0,5 là điểm làm việc mà mô hình dùng nếu không ai hiệu
        # chuẩn lại. Đo nó để biết mức lợi của bước hiệu chuẩn là bao nhiêu.
        f1_default = f1_at_threshold(labels[half_b], scores[half_b], 0.5)[0]
        oracle = best_threshold(labels, scores)
        f1_oracle = f1_at_threshold(labels, scores, oracle)[0]

        key = (run_name, dataset_name)
        ood_results[key] = {"labels": labels, "scores": scores,
                            "threshold": tuned, "trivial": trivial}
        ood_rows.append({
            "run_name": run_name, "dataset": dataset_name,
            "flow_count": int(metrics["n_flows"]),
            "positive_rate": positive_rate,
            "missing_feature_count": int(metrics["missing_features_count"]),
            "trivial_f1_baseline": trivial,
            "f1_default_threshold": f1_default,
            "f1_calibrated": f1_tuned,
            "f1_oracle": f1_oracle,
            "precision": precision, "recall": recall,
            "roc_auc": metrics["roc_auc"], "pr_auc": metrics["pr_auc"],
            "threshold_calibrated": tuned,
            "exceeds_trivial_baseline": bool(f1_tuned > trivial),
            "sample_seconds": sample_seconds,
            "inference_seconds": inference_seconds,
        })

        print(f"{run_name} trên {dataset_name}")
        print(f"  đặc trưng thiếu   {metrics['missing_features_count']}")
        print(f"  tỷ lệ tấn công    {positive_rate * 100:8.4f}%")
        print(f"  mốc tầm thường    {trivial * 100:8.4f}%")
        print(f"  F1 ngưỡng mặc định{f1_default * 100:8.4f}%")
        print(f"  F1 sau hiệu chuẩn {f1_tuned * 100:8.4f}%")
        print(f"  ROC-AUC           {metrics['roc_auc']:8.4f}")
        verdict = "vượt mốc" if f1_tuned > trivial else "không vượt mốc"
        print(f"  kết luận: {verdict} ({(f1_tuned - trivial) * 100:+.4f} điểm)")
        print(f"  thời gian: lấy mẫu {sample_seconds:.1f} giây, "
              f"suy luận {inference_seconds:.1f} giây")
        print()

# Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
# giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
# vào tệp nộp.
_written_path = fig_lib.export_table(ood_rows, cell_dir(CELL, "tables"), "ngoai_phan_phoi")
_written_manifest = fig_lib.write_manifest(
    cell_dir(CELL, "metrics"),
    {"target_rows": TARGET_ROWS, "sample_seed": SAMPLE_SEED,
     "split_seed": SPLIT_SEED,
     "protocol": "chọn ngưỡng trên nửa A, chấm điểm trên nửa B",
     "results": ood_rows}, name="ngoai_phan_phoi")

stage2_nf_cse_cic_ids2018_v3 trên NF-ToN-IoT-v3
  đặc trưng thiếu   0
  tỷ lệ tấn công     38.9824%
  mốc tầm thường     56.0969%
  F1 ngưỡng mặc định  0.8827%
  F1 sau hiệu chuẩn  69.4625%
  ROC-AUC             0.7721
  kết luận: vượt mốc (+13.3657 điểm)
  thời gian: lấy mẫu 1.4 giây, suy luận 2.3 giây



stage2_nf_unsw_nb15_v2 trên NF-ToN-IoT-v3
  đặc trưng thiếu   0
  tỷ lệ tấn công     38.9824%
  mốc tầm thường     56.0969%
  F1 ngưỡng mặc định  0.8804%
  F1 sau hiệu chuẩn  56.0969%
  ROC-AUC             0.6259
  kết luận: không vượt mốc (+0.0000 điểm)
  thời gian: lấy mẫu 0.9 giây, suy luận 1.6 giây



stage2_nf_unsw_nb15_v3 trên NF-ToN-IoT-v3
  đặc trưng thiếu   0
  tỷ lệ tấn công     38.9824%
  mốc tầm thường     56.0969%
  F1 ngưỡng mặc định 41.7907%
  F1 sau hiệu chuẩn  74.0733%
  ROC-AUC             0.8258
  kết luận: vượt mốc (+17.9764 điểm)
  thời gian: lấy mẫu 0.9 giây, suy luận 1.6 giây



## 5. Biểu đồ đánh giá out-of-distribution

Bốn loại biểu đồ cho mỗi cặp mô hình và bộ dữ liệu.

**Phân bố anomaly score** là hình quan trọng nhất của mục này. Nó cho thấy
trực tiếp hai lớp có tách nhau hay không. Nếu hai phân bố chồng lên nhau thì
không ngưỡng nào cứu được, còn nếu chúng tách rời mà kết quả vẫn kém thì vấn đề
nằm ở vị trí đặt ngưỡng chứ không ở năng lực phân biệt.

**Đường ROC và đường Precision-Recall đặt cạnh nhau.** Diện tích dưới đường ROC
không phụ thuộc tỷ lệ lớp nên so sánh được giữa hai bộ có mức mất cân bằng khác
nhau; chính vì vậy nó che mất ảnh hưởng của dương tính giả, mà đường
Precision-Recall thì phản ánh đúng.

**Quét ngưỡng** cho thấy điểm F1 biến thiên thế nào theo ngưỡng, và ngưỡng đã
chọn nằm ở đâu trên đường cong đó.

**Biểu đồ hiệu chuẩn** đối chiếu xác suất mô hình đưa ra với tần suất thực tế.
Trên dữ liệu lạ, mô hình thường dồn điểm số về sát hai đầu khiến xác suất nó báo
ra không còn phản ánh mức chắc chắn thật.


In [5]:
CELL = "03_bieu_do_ngoai_phan_phoi"

for (run_name, dataset_name), stored in ood_results.items():
    # Mỗi cặp mô hình và bộ dữ liệu có thư mục hình riêng để không đè lên nhau.
    safe_name = f"{run_name}__{dataset_name.replace('-', '_').lower()}"
    out_dir = cell_dir(f"{CELL}/{safe_name}", "figures")
    labels, scores = stored["labels"], stored["scores"]

    # Vẽ kèm ngưỡng đã hiệu chuẩn để thấy lát cắt rơi vào đâu trên vùng chồng
    # lấn giữa hai phân bố.
    fig_lib.plot_score_distribution(labels, scores, out_dir,
                                    name="phan_bo_diem_bat_thuong",
                                    threshold=stored["threshold"])
    plt.show()
    fig_lib.plot_roc_pr(labels, scores, out_dir, name="roc_va_pr")
    plt.show()
    # Hai hình dưới đây truyền out_dir=None nên chỉ hiện trong notebook, không
    # ghi ra đĩa. Chúng là hình chẩn đoán: có ích khi đang xem xét một lần chạy,
    # nhưng không xuất hiện trong bất kỳ công trình đối sánh nào nên không thuộc
    # nhóm hình đưa vào báo cáo. Ghi cả hai ra đĩa cho ba cặp đánh giá sẽ thêm
    # sáu tệp mà báo cáo không dùng tới tệp nào.
    fig_lib.plot_threshold_sweep(labels, scores, None, name="quet_nguong")
    plt.show()
    fig_lib.plot_calibration(labels, scores, None, name="hieu_chuan")
    plt.show()

    # Ma trận nhầm lẫn dựng tại ngưỡng đã hiệu chuẩn, theo quy ước hàng là nhãn
    # thật và cột là nhãn dự đoán.
    predicted = (scores >= stored["threshold"]).astype(int)
    confusion = np.array([
        [int(np.sum((predicted == 0) & (labels == 0))),
         int(np.sum((predicted == 1) & (labels == 0)))],
        [int(np.sum((predicted == 0) & (labels == 1))),
         int(np.sum((predicted == 1) & (labels == 1)))],
    ])
    fig_lib.plot_confusion_matrix(confusion, ["Lành tính", "Tấn công"],
                                  out_dir, name="ma_tran_nham_lan")
    plt.show()

    print(f"{run_name} trên {dataset_name}: 3 hình ghi ra đĩa, 2 hình chẩn đoán chỉ hiện tại đây")

stage2_nf_cse_cic_ids2018_v3 trên NF-ToN-IoT-v3: 3 hình ghi ra đĩa, 2 hình chẩn đoán chỉ hiện tại đây


stage2_nf_unsw_nb15_v2 trên NF-ToN-IoT-v3: 3 hình ghi ra đĩa, 2 hình chẩn đoán chỉ hiện tại đây


stage2_nf_unsw_nb15_v3 trên NF-ToN-IoT-v3: 3 hình ghi ra đĩa, 2 hình chẩn đoán chỉ hiện tại đây


## 6. Giải thích quyết định của mô hình

Phần này đo mức đóng góp của từng đặc trưng vào quyết định của mô hình bằng phép
ablation thuộc tính: lần lượt đặt từng cột về giá trị không rồi đo độ lệch của
xác suất tấn công so với khi giữ nguyên. Cột nào làm xác suất lệch nhiều thì cột
đó đóng góp nhiều.

Phép đo này thực hiện trên chính mô hình đã huấn luyện chứ không dựa trên một
bảng xếp hạng dựng sẵn. Giá trị thu được vì vậy thay đổi theo tập dữ liệu đang
xét, đúng như bản chất của nó: độ quan trọng của một đặc trưng không phải thuộc
tính cố định của mô hình mà là quan hệ giữa mô hình và dữ liệu.

Một hạn chế cần nêu khi diễn giải: độ quan trọng đo trên dữ liệu cùng phân phối
không nhất thiết chuyển sang dữ liệu out-of-distribution. Bảng xếp hạng dưới đây mô
tả hành vi của mô hình trên chính tập kiểm thử của nó.


In [6]:
CELL = "04_giai_thich"

from tgn_nids.experiments.ood_eval import load_checkpoint

importance_rows = []
for run_name in ANALYSIS_RUNS:
    model_path = models[run_name]["model_path"]
    checkpoint, memory, embedding, detector, device = load_checkpoint(
        model_path, device="cpu")
    feature_names = [name.strip()
                     for name in checkpoint["preprocessor_state"]["feature_names"]]

    # Lấy một tập con nhỏ của dữ liệu ngoài để đo. Số cạnh đủ lớn để trung bình
    # ổn định nhưng đủ nhỏ để phép triệt tiêu từng cột trong 49 cột chạy nhanh.
    dataset_path = next(iter(OOD_DATASETS.values()))
    if not dataset_path.exists():
        print(f"Bỏ qua {run_name}: không có dữ liệu để đo")
        continue
    subset = reservoir_sample_parquet(str(dataset_path), target_rows=20_000,
                                      seed=SAMPLE_SEED)

    # Đo bằng chính hàm đánh giá: chạy một lần với dữ liệu nguyên vẹn để lấy mốc,
    # rồi chạy lại 49 lần, mỗi lần triệt tiêu một cột.
    base = evaluate_ood_binary(model_path, subset.copy(), device="cpu",
                               return_scores=True)
    base_scores = np.asarray(base["scores"], dtype=float)

    deltas = {}
    for column in feature_names:
        if column not in subset.columns:
            # Cột vắng mặt vốn đã bị điền giá trị thay thế, nên triệt tiêu nó
            # không thay đổi gì; ghi nhận là không đo được thay vì ghi số không.
            continue
        perturbed = subset.copy()
        perturbed[column] = 0.0
        result = evaluate_ood_binary(model_path, perturbed, device="cpu",
                                     return_scores=True)
        shifted = np.asarray(result["scores"], dtype=float)
        deltas[column] = float(np.mean(np.abs(shifted - base_scores)))

    total_shift = sum(deltas.values())
    if total_shift <= 0:
        print(f"Bỏ qua {run_name}: không cột nào làm xác suất thay đổi")
        continue

    # Quy về tỷ lệ phần trăm trên tổng độ lệch, để bảng đọc được như phần đóng
    # góp tương đối thay vì một đại lượng không có đơn vị.
    importance = {column: value / total_shift for column, value in deltas.items()}
    out_dir = cell_dir(f"{CELL}/{run_name}", "figures")
    fig_lib.plot_feature_importance(importance, out_dir,
                                    name="do_quan_trong_dac_trung")
    plt.show()

    for rank, (column, value) in enumerate(
            sorted(importance.items(), key=lambda item: -item[1]), start=1):
        importance_rows.append({"run_name": run_name, "rank": rank,
                                "feature_name": column,
                                "relative_importance": value,
                                "mean_absolute_shift": deltas[column]})

    print(f"{run_name}: đo trên {len(deltas)} đặc trưng, "
          f"{len(subset):,} luồng")
    for row in importance_rows[-5:]:
        print(f"   {row['rank']:>2}. {row['feature_name']:<32}"
              f"{row['relative_importance'] * 100:7.3f}%")
    print()

# Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
# giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
# vào tệp nộp.
_written_path = fig_lib.export_table(importance_rows, cell_dir(CELL, "tables"),
                     "do_quan_trong_dac_trung")

stage2_nf_cse_cic_ids2018_v3: đo trên 49 đặc trưng, 20,000 luồng
   45. SRC_TO_DST_IAT_MAX                0.014%
   46. OUT_PKTS                          0.010%
   47. IN_PKTS                           0.009%
   48. OUT_BYTES                         0.005%
   49. FTP_COMMAND_RET_CODE              0.000%



stage2_nf_unsw_nb15_v2: đo trên 41 đặc trưng, 20,000 luồng
   37. RETRANSMITTED_IN_BYTES            0.000%
   38. RETRANSMITTED_OUT_PKTS            0.000%
   39. RETRANSMITTED_OUT_BYTES           0.000%
   40. SRC_TO_DST_SECOND_BYTES           0.000%
   41. DST_TO_SRC_SECOND_BYTES           0.000%



stage2_nf_unsw_nb15_v3: đo trên 49 đặc trưng, 20,000 luồng
   45. RETRANSMITTED_IN_PKTS             0.006%
   46. OUT_BYTES                         0.002%
   47. RETRANSMITTED_OUT_PKTS            0.001%
   48. RETRANSMITTED_IN_BYTES            0.000%
   49. RETRANSMITTED_OUT_BYTES           0.000%



## 7. Đồ thị con giải thích quanh đỉnh bị cảnh báo

Mục 6 trả lời câu hỏi *đặc trưng nào* đẩy mô hình tới quyết định. Mục này trả
lời câu hỏi còn lại mà một người vận hành cần: *cảnh báo nằm ở đâu trong mạng*.

Bộ trích xuất lấy toàn bộ đỉnh và cạnh trong phạm vi k bước quanh một đỉnh, tức
Ego-Network của đỉnh đó, rồi đánh dấu những cặp máy có luồng bị mô hình xếp là
tấn công. Đỉnh trung tâm chọn theo số luồng bị cảnh báo nhiều nhất, vì đó là
đỉnh mà người vận hành sẽ mở ra xem đầu tiên.

Phép đo chạy trên **tập kiểm thử của chính bộ dữ liệu mà mô hình được huấn
luyện**, không chạy trên bộ out-of-distribution. Lý do nằm ở định danh đỉnh: mọi địa
chỉ của tập kiểm thử đều có mặt trong ánh xạ đỉnh đã học, nên mỗi đỉnh trên hình
là một máy có thật với đúng vector nhớ mà mô hình đã xây trong lúc huấn luyện.
Trên một bộ dữ liệu khác, các địa chỉ chưa từng gặp phải dồn vào những ô nhớ có
sẵn, nhiều máy không liên quan bị nhập làm một đỉnh, và hình thu được mô tả một
cấu trúc do phép băm tạo ra chứ không phải cấu trúc của mạng.

Ràng buộc đó giới hạn phạm vi của mục này: chỉ những lần chạy có phép chia dựng
lại được chính xác từ dữ liệu gốc mới vẽ được hình. Lần chạy nào không thoả thì
bỏ qua kèm lý do, thay vì vẽ trên một tập dữ liệu không đúng.

Nhiều luồng nối cùng một cặp máy được gộp thành một cạnh, bề rộng theo logarit
số luồng. Cạnh mã hoá đồng thời bằng màu, bề rộng và kiểu nét để hình còn đọc
được khi in đen trắng.


In [7]:
CELL = "07_do_thi_con_giai_thich"

from tgn_nids.explainers.subgraph_extractor import SubgraphExtractor

# Bán kính vùng lân cận. Một bước đã đủ cho hình đưa vào báo cáo: NF-UNSW-NB15
# chỉ có vài chục máy chủ, nên hai bước sẽ kéo theo gần như toàn bộ đồ thị.
K_HOPS = 1

# Số luồng lấy từ tập kiểm thử. Giữ nhỏ để đồ thị con còn đọc được: mỗi cặp máy
# thường mang hàng nghìn luồng, và số cặp mới là thứ quyết định hình có rối hay
# không.
SUBGRAPH_FLOW_BUDGET = 4_000

# Bộ dữ liệu gốc của từng lần chạy, kèm cách chia đã dùng. Chỉ dựng lại được
# phép chia theo trình tự thời gian, vì nó chỉ phụ thuộc cột thời gian và tỷ lệ
# cắt. Phép chia phân tầng phụ thuộc bộ sinh số ngẫu nhiên bên trong ô chia của
# notebook huấn luyện nên không tái lập từ đây.
SUBGRAPH_SOURCES = {
    "stage2_nf_unsw_nb15_v3": {
        "path": PROJECT_ROOT / "data/raw/nf-unsw-nb15-v3/NF-UNSW-NB15-v3.parquet",
        "time_column": "FLOW_START_MILLISECONDS",
    },
}

subgraph_rows = []
for run_name in ANALYSIS_RUNS:
    source = SUBGRAPH_SOURCES.get(run_name)
    if source is None:
        protocol = models[run_name]["result"]["config"]["protocol"]
        print(f"Bỏ qua {run_name}: phép chia '{protocol}' không dựng lại được "
              f"từ dữ liệu gốc, nên không xác định được tập kiểm thử")
        continue
    if not source["path"].exists():
        print(f"Bỏ qua {run_name}: không có tệp {source['path'].name}")
        continue

    model_path = models[run_name]["model_path"]
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
    node_mapping = dict(checkpoint["node_mapping"])

    # Dựng lại đúng tập kiểm thử: sắp theo thời gian rồi lấy 10% cuối.
    frame = pd.read_parquet(source["path"])
    ordered = frame.sort_values(source["time_column"], kind="mergesort")
    test_frame = ordered.iloc[int(len(ordered) * 0.9):]

    # Đối chiếu với tỷ lệ tấn công đã lưu, để chắc chắn tập dựng lại đúng là tập
    # đã sinh ra các chỉ số được báo cáo.
    recorded = models[run_name]["result"]["split_stats"]["test"]
    rebuilt_rate = float(test_frame["Label"].mean())
    recorded_rate = recorded["attacks"] / recorded["count"]
    if abs(rebuilt_rate - recorded_rate) > 1e-4:
        raise RuntimeError(
            f"Tập kiểm thử dựng lại cho {run_name} có tỷ lệ tấn công "
            f"{rebuilt_rate:.6f}, còn kết quả đã lưu ghi {recorded_rate:.6f}. "
            f"Phép chia không khớp nên không dùng được.")

    subset = test_frame.sample(n=min(SUBGRAPH_FLOW_BUDGET, len(test_frame)),
                               random_state=SAMPLE_SEED).sort_index()

    # Mọi địa chỉ của tập kiểm thử đều phải có trong ánh xạ đã học; nếu không,
    # đỉnh sẽ phải mượn ô nhớ của đỉnh khác và hình mất ý nghĩa.
    addresses = pd.unique(pd.concat(
        [subset["IPV4_SRC_ADDR"].astype(str),
         subset["IPV4_DST_ADDR"].astype(str)], ignore_index=True))
    unknown = [a for a in addresses if a not in node_mapping]
    if unknown:
        print(f"Bỏ qua {run_name}: {len(unknown)} địa chỉ không có ô nhớ đã học")
        continue

    evaluation = evaluate_ood_binary(model_path, subset.copy(), device="cpu",
                                     return_scores=True)
    predictions = np.asarray(evaluation["predictions"], dtype=int)

    source_ids = subset["IPV4_SRC_ADDR"].astype(str).map(node_mapping).to_numpy()
    target_ids = subset["IPV4_DST_ADDR"].astype(str).map(node_mapping).to_numpy()
    edge_index = torch.tensor(np.vstack([source_ids, target_ids]),
                              dtype=torch.long)

    flagged = predictions == 1
    if not flagged.any():
        print(f"Bỏ qua {run_name}: không luồng nào bị xếp là tấn công")
        continue
    center_id = int(np.bincount(
        np.concatenate([source_ids[flagged], target_ids[flagged]])).argmax())
    id_to_address = {index: address for address, index in node_mapping.items()}
    center_address = id_to_address.get(center_id)

    extractor = SubgraphExtractor(k_hops=K_HOPS)
    subgraph = extractor.extract_ego_network(
        target_node_id=center_id,
        edge_index=edge_index,
        edge_predictions=torch.tensor(predictions, dtype=torch.long))

    out_dir = cell_dir(f"{CELL}/{run_name}", "figures")
    fig_lib.plot_ego_network(subgraph, out_dir, name="do_thi_con_giai_thich",
                             center_ip=center_address)
    plt.show()

    alert_flows = sum(1 for edge in subgraph["edges"] if edge["is_anomaly"])
    pairs = {tuple(sorted((edge["source"], edge["target"])))
             for edge in subgraph["edges"]}
    alert_pairs = {tuple(sorted((edge["source"], edge["target"])))
                   for edge in subgraph["edges"] if edge["is_anomaly"]}
    subgraph_rows.append({
        "run_name": run_name,
        "data_source": "tập kiểm thử của chính bộ huấn luyện",
        "k_hops": K_HOPS,
        "flows_scored": int(len(subset)),
        "center_node_id": center_id,
        "center_address": center_address,
        "subgraph_nodes": subgraph["num_nodes"],
        "subgraph_host_pairs": len(pairs),
        "subgraph_alert_pairs": len(alert_pairs),
        "subgraph_flows": subgraph["num_edges"],
        "subgraph_alert_flows": alert_flows,
    })

    print(f"{run_name}")
    print(f"  đỉnh trung tâm : {center_id}"
          f"{f' ({center_address})' if center_address else ''}")
    print(f"  đồ thị con     : {subgraph['num_nodes']} đỉnh, "
          f"{len(pairs)} cặp máy ({len(alert_pairs)} cặp có cảnh báo)")
    print(f"  luồng          : {subgraph['num_edges']:,} "
          f"({alert_flows:,} bị cảnh báo)")

# Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
# giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
# vào tệp nộp.
_written_path = fig_lib.export_table(subgraph_rows, cell_dir(CELL, "tables"),
                     "do_thi_con_giai_thich")

Bỏ qua stage2_nf_cse_cic_ids2018_v3: phép chia 'graphids' không dựng lại được từ dữ liệu gốc, nên không xác định được tập kiểm thử
Bỏ qua stage2_nf_unsw_nb15_v2: phép chia 'graphids' không dựng lại được từ dữ liệu gốc, nên không xác định được tập kiểm thử


stage2_nf_unsw_nb15_v3
  đỉnh trung tâm : 15 (175.45.176.2)
  đồ thị con     : 11 đỉnh, 10 cặp máy (10 cặp có cảnh báo)
  luồng          : 97 (97 bị cảnh báo)


## 8. Trọng số giải thích trên từng cạnh bằng GNNExplainer

Mục 7 vẽ vùng lân cận và đánh dấu cạnh nào bị mô hình xếp là tấn công. Mục này
đi thêm một bước: hỏi **cạnh nào giữ quyết định của mô hình**.

GNNExplainer học một trọng số trong khoảng 0 tới 1 cho mỗi cạnh, sao cho khi hạ
bớt các cạnh đi mà dự đoán vẫn giữ nguyên. Cạnh bị hạ về gần 0 là cạnh không cần
thiết; cạnh được giữ ở mức cao là cạnh mang thông tin quyết định. Hàm mất mát
gồm ba phần: giữ xác suất bám sát giá trị gốc, phạt để mask không giữ hết mọi
cạnh, và phạt entropy để từng trọng số dứt khoát về 0 hoặc 1 thay vì đứng lưng
chừng.

Đây là phép giải thích **hậu nghiệm**: trọng số của mô hình bị đóng băng, thứ
duy nhất được học là mask. Không có bước huấn luyện nào ở đây, và ô mã kiểm
lại điều đó bằng cách so trọng số trước và sau.

Hai điều phải nêu khi đọc kết quả. Thứ nhất, mask khởi tạo ngẫu nhiên nên hạt
giống được cố định để hai lần chạy cho cùng một hình. Thứ hai, trên đồ thị mà
mỗi đỉnh có hàng trăm lân cận, bỏ một cạnh gần như không đổi giá trị tổng hợp,
nên mask có thể phân tán khá đều; đó là tính chất của dữ liệu chứ không phải
lỗi thuật toán, và bản thân nó là một kết quả đáng báo cáo.


In [8]:
CELL = "08_trong_so_canh_gnnexplainer"

from tgn_nids.explainers import GNNExplainer

# Số vòng tối ưu mặt nạ. Đủ để mặt nạ tách hẳn thành hai cụm cao và thấp mà vẫn
# chạy trong vài giây cho mỗi luồng trên CPU.
GNNEXPLAINER_EPOCHS = 80

# Hạt giống khởi tạo mặt nạ. Cố định để hai lần chạy cho cùng một hình.
GNNEXPLAINER_SEED = 7

gnnexplainer_rows = []
for run_name in ANALYSIS_RUNS:
    source = SUBGRAPH_SOURCES.get(run_name)
    if source is None or not source["path"].exists():
        print(f"Bỏ qua {run_name}: cùng lý do đã nêu ở mục trên")
        continue

    model_path = models[run_name]["model_path"]
    checkpoint, memory, embedding, detector, _ = load_checkpoint(model_path,
                                                                 device="cpu")
    node_mapping = dict(checkpoint["node_mapping"])
    preprocessor = NetFlowPreprocessor.from_state_dict(
        checkpoint["preprocessor_state"])

    # Dựng lại đúng tập kiểm thử như mục trên, rồi lấy một mẫu nhỏ: mỗi vòng
    # tối ưu chạy lại khối nhúng trên toàn bộ đồ thị con nên chi phí tăng theo
    # số cạnh.
    frame = pd.read_parquet(source["path"])
    ordered = frame.sort_values(source["time_column"], kind="mergesort")
    test_frame = ordered.iloc[int(len(ordered) * 0.9):]
    subset = test_frame.sample(n=min(600, len(test_frame)),
                               random_state=SAMPLE_SEED).sort_index()

    features = preprocessor.transform(subset)[preprocessor.feature_names].values
    features = np.nan_to_num(features.astype(np.float32), nan=0.0,
                             posinf=1e4, neginf=-1e4)
    source_ids = subset["IPV4_SRC_ADDR"].astype(str).map(node_mapping).to_numpy()
    target_ids = subset["IPV4_DST_ADDR"].astype(str).map(node_mapping).to_numpy()
    edge_index = torch.tensor(np.vstack([source_ids, target_ids]),
                              dtype=torch.long)
    edge_attr = torch.tensor(features)

    # Chọn luồng để giải thích: luồng mà mô hình cho xác suất tấn công cao nhất.
    # Giải thích một luồng mà mô hình vốn không cảnh báo thì không có quyết định
    # nào để giải thích.
    benign_class_id = checkpoint["benign_class_id"]
    with torch.no_grad():
        node_memory = memory(torch.arange(checkpoint["n_nodes"]))[0]
        z = embedding(node_memory, edge_index, edge_attr)
        attack_probability = 1.0 - torch.softmax(
            detector(z[edge_index[0]], z[edge_index[1]], edge_attr),
            dim=1)[:, benign_class_id]
    target_index = int(attack_probability.argmax())

    # Ảnh chụp trọng số trước khi giải thích, để chứng minh phép giải thích này
    # không đụng vào mô hình.
    weights_before = {name: tensor.clone()
                      for name, tensor in detector.state_dict().items()}

    explainer = GNNExplainer(memory, embedding, detector, device="cpu",
                             benign_class_id=benign_class_id,
                             epochs=GNNEXPLAINER_EPOCHS)
    explanation = explainer.explain_edge(
        target_index, node_memory, edge_index, edge_attr,
        preprocessor.feature_names, seed=GNNEXPLAINER_SEED)

    changed = sum(1 for name, tensor in detector.state_dict().items()
                  if not torch.equal(weights_before[name], tensor))
    if changed:
        raise RuntimeError(
            f"GNNExplainer đã làm đổi {changed} tham số của mô hình. Phép giải "
            f"thích hậu nghiệm không được phép sửa mô hình.")

    edge_mask = explanation["edge_mask"]

    # Vẽ lại đồ thị con quanh đúng luồng vừa giải thích, bề rộng cạnh nay theo
    # trọng số giải thích thay vì theo số luồng.
    center_id = int(edge_index[0, target_index])
    extractor = SubgraphExtractor(k_hops=K_HOPS)
    subgraph = extractor.extract_ego_network(
        target_node_id=center_id, edge_index=edge_index,
        edge_predictions=(attack_probability > 0.5).long())

    # Mặt nạ tính trên toàn bộ cạnh của lô; lọc lấy đúng các cạnh nằm trong đồ
    # thị con để hai danh sách khớp thứ tự.
    in_subgraph = [i for i in range(edge_index.shape[1])
                   if int(edge_index[0, i]) in
                   {n["original_id"] for n in subgraph["nodes"]}
                   and int(edge_index[1, i]) in
                   {n["original_id"] for n in subgraph["nodes"]}]
    weights_in_subgraph = [float(edge_mask[i]) for i in in_subgraph]

    id_to_address = {index: address for address, index in node_mapping.items()}
    out_dir = cell_dir(f"{CELL}/{run_name}", "figures")
    fig_lib.plot_ego_network(subgraph, out_dir, name="trong_so_canh_giai_thich",
                             center_ip=id_to_address.get(center_id),
                             edge_weights=weights_in_subgraph)
    plt.show()

    gnnexplainer_rows.append({
        "run_name": run_name,
        "flows_scored": int(len(subset)),
        "target_index": target_index,
        "attack_probability_before": explanation["attack_probability_before"],
        "attack_probability_after": explanation["attack_probability_after"],
        "edge_mask_min": float(edge_mask.min()),
        "edge_mask_median": float(np.median(edge_mask)),
        "edge_mask_max": float(edge_mask.max()),
        "edge_mask_std": float(edge_mask.std()),
        "edges_kept_above_half": int((edge_mask > 0.5).sum()),
        "edges_total": int(len(edge_mask)),
        "model_parameters_changed": changed,
        "epochs": GNNEXPLAINER_EPOCHS,
    })

    row = gnnexplainer_rows[-1]
    print(f"{run_name}")
    print(f"  luồng giải thích : {target_index}, xác suất tấn công "
          f"{row['attack_probability_before']:.6f}")
    print(f"  sau khi áp mặt nạ: {row['attack_probability_after']:.6f}")
    print(f"  mặt nạ cạnh      : nhỏ nhất {row['edge_mask_min']:.4f}, "
          f"trung vị {row['edge_mask_median']:.4f}, "
          f"lớn nhất {row['edge_mask_max']:.4f}")
    print(f"  cạnh được giữ    : {row['edges_kept_above_half']}/"
          f"{row['edges_total']} có trọng số trên 0,5")
    print(f"  tham số mô hình bị đổi: {changed}")

# Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
# giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
# vào tệp nộp.
_written_path = fig_lib.export_table(gnnexplainer_rows, cell_dir(CELL, "tables"),
                                     "trong_so_canh_giai_thich")

Bỏ qua stage2_nf_cse_cic_ids2018_v3: cùng lý do đã nêu ở mục trên
Bỏ qua stage2_nf_unsw_nb15_v2: cùng lý do đã nêu ở mục trên


stage2_nf_unsw_nb15_v3
  luồng giải thích : 356, xác suất tấn công 0.999894
  sau khi áp mặt nạ: 0.986473
  mặt nạ cạnh      : nhỏ nhất 0.1082, trung vị 0.1225, lớn nhất 0.8978
  cạnh được giữ    : 189/600 có trọng số trên 0,5
  tham số mô hình bị đổi: 0


## 9. Chiếu không gian nhúng theo loại tấn công

Hình định tính, trả lời câu hỏi mà mọi chỉ số định lượng đều không trả lời: bộ
mã hoá có tách các họ tấn công thành cụm riêng trong không gian biểu diễn hay
không, hay nó chỉ học được ranh giới lành tính với tấn công.

**Khác biệt với hình cùng loại của GraphIDS.** Nhóm GraphIDS chiếu vector nhúng
của dữ liệu **cùng phân phối** với dữ liệu huấn luyện. Hình dưới đây chiếu vector
nhúng của dữ liệu **out-of-distribution**, vì đó là dữ liệu mà notebook này nạp sẵn.
Hai hình vì vậy trả lời hai câu khác nhau: hình của họ hỏi mô hình có học được
cấu trúc lớp trên miền đã thấy không, hình này hỏi cấu trúc đó có còn giữ được
trên một miền chưa từng thấy không. Phải ghi rõ điều này khi đưa vào báo cáo,
không đặt hai hình cạnh nhau như thể cùng một phép đo.

Lưu ý khi đọc: t-SNE không bảo toàn khoảng cách toàn cục, nên khoảng cách giữa
hai cụm trên hình không mang ý nghĩa định lượng. Chỉ đọc được việc các điểm cùng
nhãn có tụ lại gần nhau hay không.


In [9]:
CELL = "05_chieu_nhung"

import torch

from tgn_nids.data.preprocess import NetFlowPreprocessor

# Số cạnh đưa vào phép chiếu. t-SNE có độ phức tạp bậc hai nên vài nghìn điểm là
# ngưỡng thực dụng; hàm vẽ tự rút mẫu thêm nếu vượt max_points.
PROJECTION_ROWS = 6_000


def edge_embeddings(model_path, frame, batch_size=512):
    """Tính vector nhúng cạnh cho một bảng luồng, dùng đúng bộ mã hoá đã lưu.

    Lặp lại đúng trình tự của ood_eval.evaluate_ood_binary: nạp checkpoint, đưa
    bảng qua chính bộ tiền xử lý đã khớp lúc huấn luyện, tra địa chỉ sang chỉ số
    đỉnh theo đúng bảng ánh xạ đã lưu, rồi ghép biểu diễn hai đầu của mỗi cạnh.
    Không huấn luyện lại gì, chỉ suy luận.
    """
    checkpoint, memory, embedding, _detector, device = load_checkpoint(
        model_path, device="cpu")
    # Dựng lại bộ tiền xử lý từ bản kê tham số dạng dữ liệu thuần. Checkpoint
    # không cất nguyên đối tượng nữa, để tệp nạp được ở chế độ an toàn của
    # torch.load và mở được trên máy không cài gói mã của dự án.
    preprocessor = NetFlowPreprocessor.from_state_dict(
        checkpoint["preprocessor_state"])
    node_mapping = dict(checkpoint["node_mapping"])
    n_nodes = checkpoint["n_nodes"]

    columns = {str(c).strip().lower(): c for c in frame.columns}
    src_column = columns.get("ipv4_src_addr")
    dst_column = columns.get("ipv4_dst_addr")
    if src_column is None or dst_column is None:
        # Dừng hẳn thay vì sinh địa chỉ thay thế. Một phép chiếu dựng trên đỉnh
        # bịa trông giống hệt phép chiếu thật nhưng không nói gì về mô hình.
        raise ValueError(
            "Bảng không có cột địa chỉ nên không dựng được đỉnh đồ thị. "
            "Không sinh địa chỉ thay thế; bỏ qua phép chiếu cho bộ này.")

    processed = preprocessor.transform(frame)
    features = processed[preprocessor.feature_names].values.astype("float32")
    features = np.nan_to_num(features, nan=0.0, posinf=1e4, neginf=-1e4)

    # Địa chỉ chưa từng thấy được băm vào bảng bộ nhớ cố định, đúng cách mà
    # ood_eval làm, và bằng crc32 để kết quả lặp lại được qua các lần chạy.
    import zlib
    def node_id(address):
        address = str(address)
        if address not in node_mapping:
            node_mapping[address] = zlib.crc32(address.encode("utf-8")) % n_nodes
        return node_mapping[address]

    src_ids = frame[src_column].map(node_id).astype("int64").values
    dst_ids = frame[dst_column].map(node_id).astype("int64").values

    chunks = []
    with torch.no_grad():
        for start in range(0, len(frame), batch_size):
            end = min(start + batch_size, len(frame))
            src = torch.tensor(src_ids[start:end], dtype=torch.long)
            dst = torch.tensor(dst_ids[start:end], dtype=torch.long)
            edge_attr = torch.tensor(features[start:end], dtype=torch.float32)

            node_index = torch.unique(torch.cat([src, dst]))
            assoc = torch.full((int(node_index.max().item()) + 1,), -1,
                               dtype=torch.long)
            assoc[node_index] = torch.arange(node_index.size(0))

            mem, _ = memory(node_index)
            z = embedding(mem, torch.stack([assoc[src], assoc[dst]], dim=0), edge_attr)
            # Biểu diễn của một cạnh là ghép biểu diễn hai đầu, đúng thứ mà đầu
            # phân loại nhận vào.
            chunks.append(torch.cat([z[assoc[src]], z[assoc[dst]]], dim=1).numpy())

            memory.update_state(src, dst,
                                torch.arange(start, end, dtype=torch.long), edge_attr)
            memory.detach_memory()
    return np.concatenate(chunks, axis=0)


projection_rows = []
for run_name in ANALYSIS_RUNS:
    for dataset_name, dataset_path in OOD_DATASETS.items():
        if not dataset_path.exists():
            continue
        sample = reservoir_sample_parquet(str(dataset_path),
                                          target_rows=PROJECTION_ROWS,
                                          seed=SAMPLE_SEED)
        try:
            vectors = edge_embeddings(models[run_name]["model_path"], sample.copy())
        except ValueError as reason:
            print(f"Bỏ qua {run_name} trên {dataset_name}: {reason}")
            continue

        attack_column = next((c for c in sample.columns
                              if str(c).strip().lower() == "attack"), None)
        if attack_column is None:
            print(f"Bỏ qua {run_name} trên {dataset_name}: không có cột nhãn đa lớp")
            continue
        names = sorted(sample[attack_column].astype(str).unique())
        codes = sample[attack_column].astype(str).map(
            {name: index for index, name in enumerate(names)}).values

        pair = f"{run_name}__{dataset_name.lower().replace('-', '_')}"
        fig_lib.plot_embedding_projection(
            vectors, codes, names, cell_dir(CELL, "figures"), name=pair)
        plt.show()
        projection_rows.append({"run_name": run_name, "dataset": dataset_name,
                                "n_edges": int(len(vectors)),
                                "embedding_dim": int(vectors.shape[1]),
                                "n_classes": len(names)})

if projection_rows:
    # Gán vào biến để Jupyter không hiển thị giá trị trả về của lệnh cuối ô:
    # giá trị đó là đường dẫn tuyệt đối, sẽ lộ bố cục ổ đĩa của máy đã chạy
    # vào tệp nộp.
    _written_path = fig_lib.export_table(projection_rows, cell_dir(CELL, "tables"), "chieu_nhung")
    print(f"{'Lần chạy':<34}{'Bộ ngoài':<18}{'Số cạnh':>9}{'Chiều':>8}{'Số lớp':>8}")
    for row in projection_rows:
        print(f"{row['run_name']:<34}{row['dataset']:<18}{row['n_edges']:>9,}"
              f"{row['embedding_dim']:>8}{row['n_classes']:>8}")
else:
    print("Không vẽ được phép chiếu nào.")

Lần chạy                          Bộ ngoài            Số cạnh   Chiều  Số lớp
stage2_nf_cse_cic_ids2018_v3      NF-ToN-IoT-v3         6,000     384      10
stage2_nf_unsw_nb15_v2            NF-ToN-IoT-v3         6,000     384      10
stage2_nf_unsw_nb15_v3            NF-ToN-IoT-v3         6,000     384      10


## 10. Bảng tổng hợp

Bảng cuối cùng gom kết quả trên tập kiểm thử cùng phân phối và kết quả ngoài
phân phối vào một chỗ. Bảng này là nguồn số liệu chính thức cho phần đánh giá khả
năng khái quát trong báo cáo.

Hai nhóm chỉ số phải đọc tách bạch. Chỉ số trên tập kiểm thử đo năng lực trong
điều kiện mà mô hình đã học; chỉ số out-of-distribution đo phần năng lực còn giữ
được khi rời khỏi điều kiện đó. Đặt hai con số cạnh nhau trong cùng một câu mà
không nêu rõ điều kiện là cách trình bày gây hiểu sai.


In [10]:
CELL = "06_tong_hop"

summary_rows = []
for run_name in ANALYSIS_RUNS:
    result = models[run_name]["result"]
    metrics = result["metrics"]
    # Chỉ số trên tập kiểm thử cùng phân phối, đọc lại từ kết quả huấn luyện.
    summary_rows.append({
        "run_name": run_name,
        "evaluation": "tập kiểm thử cùng phân phối",
        "dataset": result["config"].get("description", ""),
        "f1_macro": metrics.get("f1_macro"),
        "accuracy": metrics.get("accuracy"),
        "pr_auc": metrics.get("pr_auc"),
        "roc_auc": metrics.get("roc_auc"),
        "trivial_f1_baseline": None,
        "exceeds_trivial_baseline": None,
        "missing_feature_count": 0,
    })

# Chỉ số ngoài phân phối, đọc từ bảng đã dựng ở phần trước.
for row in ood_rows:
    summary_rows.append({
        "run_name": row["run_name"],
        "evaluation": "ngoài phân phối",
        "dataset": row["dataset"],
        "f1_macro": row["f1_calibrated"],
        "accuracy": None,
        "pr_auc": row["pr_auc"],
        "roc_auc": row["roc_auc"],
        "trivial_f1_baseline": row["trivial_f1_baseline"],
        "exceeds_trivial_baseline": row["exceeds_trivial_baseline"],
        "missing_feature_count": row["missing_feature_count"],
    })

fig_lib.export_table(summary_rows, cell_dir(CELL, "tables"), "tong_hop_phan_tich")
fig_lib.write_manifest(cell_dir(CELL, "metrics"),
                       {"variant": VARIANT,
                        "source_version": tgn_nids.SOURCE_VERSION,
                        "results": summary_rows}, name="tong_hop_phan_tich")

# Hình tổng hợp: Macro F1 của từng mô hình dưới hai điều kiện đánh giá
#
# Ô này trước đây chỉ xuất bảng. Bảng đọc được từng dòng nhưng không cho thấy
# khoảng cách giữa hai điều kiện, mà chính khoảng cách đó mới là kết luận của
# phần phân tích: mô hình giữ được bao nhiêu năng lực khi rời khỏi phân phối đã
# học. Cột màu nhấn là điều kiện cùng phân phối, cột xám là ngoài phân phối.
in_distribution = {}
out_of_distribution = {}
for row in summary_rows:
    # Rút gọn tên cho nhãn trục đỡ dài, bỏ tiền tố chung "stage2_".
    short_name = row["run_name"].replace("stage2_", "")
    if row["evaluation"] == "ngoài phân phối":
        out_of_distribution[f"{short_name}\n(ngoài phân phối)"] = {"f1_macro": row["f1_macro"]}
    else:
        in_distribution[f"{short_name}\n(cùng phân phối)"] = row["f1_macro"]

# Mốc tầm thường lấy từ các dòng ngoài phân phối, nơi đại lượng này tính được.
# Mọi cột nằm dưới đường mốc nghĩa là mô hình thua cả việc gán nhãn tấn công cho
# mọi mẫu, nên đường này phải có mặt để con số không bị đọc rời ngữ cảnh.
trivial_marks = [row["trivial_f1_baseline"] for row in summary_rows
                 if row["trivial_f1_baseline"] is not None]
fig_lib.plot_baseline_comparison(
    in_distribution, out_of_distribution, metric="f1_macro",
    out_dir=cell_dir(CELL, "figures"), name="tong_hop_macro_f1",
    trivial=max(trivial_marks) if trivial_marks else None,
    ylabel="Macro F1 (%)")
plt.show()

print(f"{'Lần chạy':<30}{'Điều kiện đánh giá':<30}{'Macro F1':>10}{'Mốc':>10}")
for row in summary_rows:
    trivial = ("—" if row["trivial_f1_baseline"] is None
               else f"{row['trivial_f1_baseline'] * 100:8.3f}%")
    print(f"{row['run_name']:<30}{row['evaluation']:<30}"
          f"{row['f1_macro'] * 100:>9.3f}%{trivial:>10}")

Lần chạy                      Điều kiện đánh giá              Macro F1       Mốc
stage2_nf_cse_cic_ids2018_v3  tập kiểm thử cùng phân phối      56.372%         —
stage2_nf_unsw_nb15_v2        tập kiểm thử cùng phân phối      53.340%         —
stage2_nf_unsw_nb15_v3        tập kiểm thử cùng phân phối      55.961%         —
stage2_nf_cse_cic_ids2018_v3  ngoài phân phối                  69.463%   56.097%
stage2_nf_unsw_nb15_v2        ngoài phân phối                  56.097%   56.097%
stage2_nf_unsw_nb15_v3        ngoài phân phối                  74.073%   56.097%
